# 11.6 Broadcast Joins

## Learning Objectives

By the end of this lesson you will be able to:

- Understand what a Broadcast Join is
- Learn when to use Broadcast Joins
- Understand how Broadcast Joins improve performance
- Learn the benefits of Broadcast Joins
- Understand Broadcast Join limitations
- Identify Broadcast Joins in Execution Plans

> **Core Rule:** Broadcast Join is one of the most practical and commonly used Spark optimization techniques.

## Setup: Initialize Spark & Mock Data

Let's start our local Spark Session and generate a large `orders` dataset and a small `customers` dataset to test Broadcast Joins.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Initialize a local Spark session
spark = SparkSession.builder \
    .appName("Broadcast_Joins_Deep_Dive") \
    .master("local[*]") \
    .getOrCreate()

# Create a "Large" Orders Table (1 Million rows for local testing)
orders = spark.range(1, 1000000).withColumnRenamed("id", "order_id") \
              .withColumn("customer_id", (F.rand() * 100).cast("int"))

# Create a "Small" Customers Table (100 rows)
customers = spark.range(1, 100).withColumnRenamed("id", "customer_id") \
                 .withColumn("name", F.concat(F.lit("Customer_"), F.col("customer_id")))

print("Spark Session Initialized and Mock Data Created!")

# Why Learn Broadcast Joins?

In the previous lesson, we learned that joins are often expensive.

Many joins require:
- Data movement
- Shuffle operations
- Network communication

These operations increase execution time.

Broadcast Join helps reduce these costs by avoiding unnecessary data movement.

# Real-World Example

Suppose an e-commerce company has:

**Orders Table**
- 500 Million Rows

**Customers Table**
- 50,000 Rows

To join these datasets, should Spark move 500 million rows? Or should Spark move only 50,000 rows?

Broadcast Join solves this problem efficiently.

# What is a Broadcast Join?

A Broadcast Join is a join strategy where Spark:

1. Identifies the smaller dataset
2. Copies that dataset to all executors
3. Performs joins locally

Since the larger dataset remains in place, large-scale shuffle operations are avoided.

<h3>Broadcast Join Architecture</h3>

<img src="../assets/Module_11/11_06_01.png" alt="Broadcast Join Architecture" width="75%">

<p><i><b>Image Prompt:</b> Apache Spark Broadcast Join architecture. Small dimension table copied to all executors while large fact table remains distributed. Local joins performed without shuffle. Clean educational architecture diagram, white background.</i></p>

# Traditional Join vs Broadcast Join

**Traditional Join:**
- Both datasets participate in shuffle
- Large network traffic
- More execution time

**Broadcast Join:**
- Small table copied once
- Minimal shuffle
- Faster execution

This difference often leads to significant performance improvements.

<h3>Traditional Join vs Broadcast Join</h3>

<img src="../assets/Module_11/11_06_02.png" alt="Traditional vs Broadcast Join" width="75%">

<p><i><b>Image Prompt:</b> Side-by-side comparison. Traditional Spark join showing heavy shuffle of both datasets. Broadcast Join showing only small table copied to executors. Educational performance comparison infographic.</i></p>

# Example Scenario

Large Table: Orders (100 Million Rows)
Small Table: Customers (10,000 Rows)

Joining both tables using shuffle would be expensive. Broadcasting 10,000 rows is much cheaper.

# How Broadcast Join Works

**Step 1:** Spark identifies a small table.
**Step 2:** The table is serialized.
**Step 3:** The serialized copy is sent to every executor.
**Step 4:** Each executor performs local matching.

No large-scale redistribution of the big table is required.

<h3>Broadcast Join Workflow</h3>

<img src="../assets/Module_11/11_06_03.png" alt="Broadcast Join Workflow" width="75%">

<p><i><b>Image Prompt:</b> Step-by-step Spark Broadcast Join workflow showing small table broadcast from driver to executors followed by local joins. Beginner-friendly architecture diagram.</i></p>

# Spark's Automatic Broadcast Join

Spark can automatically choose Broadcast Join.

If a dataset is sufficiently small, Spark may broadcast it without any user intervention. This decision is made by the Catalyst optimizer.

In [ ]:
# Example join (Spark will likely auto-broadcast this because customers table is tiny)
result_auto = orders.join(customers, "customer_id")

print("Execution Plan (Checking for Automatic Broadcast):")
print("-"*50)
result_auto.explain()
print("-"*50)
print("Notice the 'BroadcastHashJoin' and 'BroadcastExchange' in the plan!")

# Auto Broadcast Threshold

Spark uses a configuration value called:
`spark.sql.autoBroadcastJoinThreshold`

This setting determines the maximum table size eligible for automatic broadcasting. If the table is below the threshold, Spark may choose Broadcast Join automatically.

# Checking the Threshold

The threshold can be viewed using:
`spark.conf.get("spark.sql.autoBroadcastJoinThreshold")`

The exact value may vary depending on Spark version and configuration. Let's check our current threshold.

In [ ]:
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

# Convert bytes to Megabytes for readability
threshold_mb = int(threshold) / (1024 * 1024)

print(f"Current Auto Broadcast Threshold: {threshold} bytes ({threshold_mb} MB)")
print("Any table smaller than this size will be automatically broadcasted by Spark.")

# Manually Forcing a Broadcast Join

Sometimes Spark does not automatically choose Broadcast Join (e.g., if it cannot estimate the table size correctly).

In such situations, engineers can provide a hint. Spark provides the `broadcast()` function.

In [ ]:
from pyspark.sql.functions import broadcast

# To simulate Spark NOT broadcasting automatically, we temporarily set the threshold to -1 (Disabled)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("=== WITHOUT BROADCAST HINT ===")
orders.join(customers, "customer_id").explain()
print("-> Spark falls back to a costly 'SortMergeJoin' and 'Exchange' (Shuffle).\n")

print("=== WITH BROADCAST HINT ===")
# Now we explicitly use the broadcast() hint to force the optimization
result_forced = orders.join(broadcast(customers), "customer_id")
result_forced.explain()
print("-> Spark uses 'BroadcastHashJoin' perfectly!")

# Reset threshold back to default 10MB
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

# Why Use broadcast()?

The `broadcast()` hint tells Spark:

*"This dataset is small enough to copy to all executors."*

Spark will attempt to use a Broadcast Hash Join. This can improve performance significantly.

<h3>Using broadcast() Hint</h3>

<img src="../assets/Module_11/11_06_04.png" alt="Broadcast Hint" width="75%">

<p><i><b>Image Prompt:</b> Spark broadcast() hint illustration. Developer applying broadcast() to a small table causing Spark optimizer to select Broadcast Hash Join. Educational coding infographic.</i></p>

# Benefits of Broadcast Join

Major benefits include:

- Reduced Shuffle
- Lower Network Traffic
- Faster Execution
- Lower Disk Usage
- Better Resource Utilization

These benefits make Broadcast Join one of the most valuable Spark optimizations.

# Benefit 1: Reduced Shuffle

Traditional joins often require data redistribution. Broadcast Join minimizes this movement.

Less shuffle means:
- Faster stages
- Reduced network overhead
- Lower execution time

<h3>Reduced Shuffle</h3>
<img src="../assets/Module_11/11_06_05.png" alt="Reduced Shuffle" width="75%">
<p><i><b>Image Prompt:</b> Comparison showing heavy shuffle in traditional join versus minimal shuffle in Broadcast Join. Spark optimization educational diagram.</i></p>

# Benefit 2: Faster Execution

Because less data moves across the network, Spark spends less time:
- Shuffling
- Sorting
- Waiting for transfers

As a result, joins complete drastically faster.

# Benefit 3: Better Scalability

Broadcast Join performs particularly well for:
- Fact tables joined with Dimension tables
- Lookup datasets

These patterns are common in data warehouses and analytics systems.

<h3>Fact and Dimension Join</h3>

<img src="../assets/Module_11/11_06_06.png" alt="Fact and Dimension Join" width="75%">

<p><i><b>Image Prompt:</b> Star schema example showing large fact table joined with small dimension table using Broadcast Join. Data warehouse educational architecture diagram.</i></p>

# When Should You Use Broadcast Join?

Broadcast Join is usually appropriate when:
- One table is very small (typically < 10MB to 50MB, depending on cluster memory)
- One table is significantly larger
- The small table fits comfortably in the memory of a single executor

This is one of the most common use cases in Spark.

# Typical Broadcast Join Candidates

Examples include:
- Customer Master Data
- Product Catalogs
- Country Codes
- Currency Tables
- Lookup Tables

These datasets are usually small and rarely change.

<h3>Good Broadcast Join Candidates</h3>

<img src="../assets/Module_11/11_06_07.png" alt="Broadcast Candidates" width="75%">

<p><i><b>Image Prompt:</b> Examples of small lookup tables including customer dimension, product dimension, country codes and currency mappings suitable for Broadcast Join. Educational infographic.</i></p>

# Limitations of Broadcast Join

Broadcast Join is not always the correct choice. Problems arise when:
- The table is too large
- Executor memory is limited
- Multiple broadcasts occur simultaneously

These situations can create memory pressure and crash your executors (`OutOfMemoryError`).

# Large Broadcast Example

Suppose a table contains: **50 Million Rows**

Broadcasting such a dataset to every executor would be incredibly expensive. Memory usage would spike across the entire cluster, and network transferring the 50 million rows would overwhelm the system.

Performance would actually become worse.

<h3>Broadcast Join Memory Problem</h3>

<img src="../assets/Module_11/11_06_08.png" alt="Memory Warning" width="75%">

<p><i><b>Image Prompt:</b> Spark executors receiving an excessively large broadcast table causing memory pressure and potential out-of-memory situations. Educational warning infographic.</i></p>

# Identifying Broadcast Joins

Execution Plans reveal whether Broadcast Join was selected.

Look for:
`BroadcastHashJoin` and `BroadcastExchange`
in the Physical Plan.

In [ ]:
# Final check of the Physical Plan to reinforce identifying the keywords
result_forced.explain(mode="formatted")

# Example Physical Plan

A plan might contain:

BroadcastHashJoin
    ↓
Scan Orders
    ↓
BroadcastExchange

These operators confirm that Spark performed broadcasting.

<h3>BroadcastHashJoin Operator</h3>
<img src="../assets/Module_11/11_06_09.png" alt="Physical Plan Operators" width="75%">
<p><i><b>Image Prompt:</b> Spark Physical Plan highlighting BroadcastHashJoin and BroadcastExchange operators. Educational execution plan visualization showing join optimization.</i></p>

# Real-World Example

**Orders Table:** 200 Million Rows
**Products Table:** 20,000 Rows

- **Without Broadcast Join:** Execution Time = 15 Minutes
- **With Broadcast Join:** Execution Time = 4 Minutes

Same hardware. Same data. Different join strategy.

# Key Takeaways

- Broadcast Join copies a small dataset to all executors.
- It minimizes shuffle operations.
- It drastically reduces network traffic and execution time.
- It works best when one table is significantly smaller.
- Large tables should not be broadcast.
- `BroadcastHashJoin` in the Physical Plan indicates broadcast optimization.

---

# Next Lesson: 11.7 Sort Merge Joins

In the next lesson we will explore:
- Execution Flow of Sort Merge Joins
- How Spark handles joining two massive datasets

Sort Merge Join is the most common join strategy used when data is simply too large to broadcast.